# Chatbot module evaluation

### Import modules and dependencies

In [ ]:
import time
import nest_asyncio
nest_asyncio.apply()

from io import StringIO

from llama_index.core import Document, PromptTemplate
from llama_index.core.schema import TextNode
from llama_index.core.schema import NodeWithScore
from llama_index.core.indices.property_graph import DynamicLLMPathExtractor

from dotenv import load_dotenv
load_dotenv("./../../../environment/.env")

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from chatbot.data.character_story import DR_CHOI_REWRITE

from chatbot.utils.translator import Translator
from chatbot.prompt.graph.extraction import (
    EXTRACT_ENTITIES_PROMPT_TEMPLATE,
    EXTRACT_GRAPH_TRIPLETS_PROMPT_TEMPLATE
)
from chatbot.prompt.instruction.summary import INSTRUCTION_SUMMARY_PROMPT
from chatbot.prompt.instruction.extraction import ASSISTANT_NAME_EXTRACTION_PROMPT
from chatbot.prompt.routing.query_routing import QUERY_ROUTING_PROMPT_TEMPLATE


from chatbot.utils.chat_store import CacheChatStore, PersistentChatStore
from chatbot.utils.generator import Generator, ResponseMode
from chatbot.utils.graph_retriever import Retriever, CustomSubRetriever
from chatbot.utils.graph_store import FalkorDBGraphStore, parse_dynamic_triplets_with_props
from chatbot.utils.models_client import EmbedderCore, LLMCore
from chatbot.utils.predefined_entities import (
    ENTITY_PROPERTIES,
    ENTITY_TYPES,
    RELATION_PROPERTIES,
    RELATION_TYPES
)
from chatbot.utils.translator import nltk
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     /home/adminilluminus/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/adminilluminus/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/adminilluminus/miniconda3/envs/chatbot2/lib/python3.9/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in LLMCore has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


### Modules initialization

In [2]:
def check_bool(text: str) -> bool:
    if text.lower() == "true":
        return True
    elif text.lower() == "false":
        return False
    else:
        raise ValueError(f"Invalid boolean value: {text}")

class Settings:
    def __init__(self):
        # Model serving settings
        self.EMBEDDER_SERVING_URL = "http://172.30.84.182:8011"
        self.LLM_SERVING_URL = "http://172.30.84.182:8013"
        self.QUANT_LARGE_MODEL_ID = os.getenv("QUANT_LARGE_MODEL_ID", "meta-llama/Meta-Llama-3.1-8B-Instruct")
        self.EMBEDDER_MODEL_ID = os.getenv("EMBEDDER_MODEL_ID", "BAAI/llm-embedder")
        self.MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", 256))
        self.LONG_MAX_NEW_TOKENS = int(os.getenv("LONG_MAX_NEW_TOKENS", 4000))
        self.USE_OPENAI_API = True
        self.OPENAI_RESPONSE_MODEL_ID = os.getenv("OPENAI_RESPONSE_MODEL_ID", "gpt-4o-mini")
        self.OPENAI_MODEL_ID = os.getenv("OPENAI_MODEL_ID", "gpt-4o-mini")
        self.OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

        # Cache chat store settings
        self.CHAT_STORE_HOST = "172.30.84.182"
        self.CHAT_STORE_PORT = int(os.getenv("CHAT_STORE_PORT", 6379))
        self.CHAT_STORE_DB = int(os.getenv("CHAT_STORE_DB", 0))
        self.CHAT_STORE_USERNAME = os.getenv("CHAT_STORE_USERNAME")
        self.CHAT_STORE_PASSWORD = os.getenv("CHAT_STORE_PASSWORD")
        self.CHAT_STORE_TTL = int(os.getenv("CHAT_STORE_TTL", 86400))
        self.CHAT_STORE_MAX_MESSAGES_PAIRS = 8

        # Persistent chat store settings
        self.MONGODB_URL = os.getenv("MONGODB_URL")
        self.PERSISTENT_CHAT_STORE_DB = os.getenv("PERSISTENT_CHAT_STORE_DB", "chatbot")
        self.PERSISTENT_CHAT_STORE_COLLECTION = os.getenv("PERSISTENT_CHAT_STORE_COLLECTION", "chat_history")

        # Graph store settings
        self.GRAPH_STORE_HOST = "172.30.84.182"
        self.GRAPH_STORE_PORT = int(os.getenv("GRAPH_STORE_PORT", 7687))
        self.GRAPH_STORE_USERNAME = os.getenv("GRAPH_STORE_USERNAME")
        self.GRAPH_STORE_PASSWORD = os.getenv("GRAPH_STORE_PASSWORD")
        self.GRAPH_STORE_BUILD = check_bool(os.getenv("GRAPH_STORE_BUILD", True))
        self.GRAPH_STORE_FORCE_BUILD = check_bool(os.getenv("GRAPH_STORE_FORCE_BUILD", False))

        # Retriever settings
        self.TOP_K_RETRIEVAL = int(os.getenv("TOP_K_RETRIEVAL", 5))
        self.PATH_DEPTH_GRAPH_RETRIEVAL = int(os.getenv("PATH_DEPTH_GRAPH_RETRIEVAL", 1))


SETTINGS = Settings()

avatar_name = "Choi"
avatar_instruction_text = ""
character_id = "Choi"
user_id = "test_user"

In [3]:
use_default_story = True
if avatar_name != "Choi" and avatar_instruction_text != "":
    user_id = user_id
    assistant_name = avatar_name
    character_id = character_id
    summarized_user_id = "the user"
    summarized_assistant_id = "the assistant"
    use_default_story = False
else:
    user_id = user_id
    assistant_name = avatar_name
    character_id = "Choi"
    summarized_user_id = "David"
    summarized_assistant_id = "Choi"

print("Initializing the translator and language detector...")
en_translator = Translator(
    source="auto", target="en", second_target="ko", capitalize_sentences=True
)
# Warm up the translator
translate_time = 0.5
max_retry = 10
while translate_time > 0.3 and max_retry > 0:
    start_translate_time = time.time()
    en_translator.translate("Warm up the translator")
    translate_time = time.time() - start_translate_time

print("Initializing the chat store...")
memory_summarize_llm = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.QUANT_LARGE_MODEL_ID,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    is_chat=False,
    use_openai=False,
)
cache_chat_store = CacheChatStore(
    host=SETTINGS.CHAT_STORE_HOST,
    port=int(SETTINGS.CHAT_STORE_PORT),
    db=int(SETTINGS.CHAT_STORE_DB),
    username=SETTINGS.CHAT_STORE_USERNAME,
    password=SETTINGS.CHAT_STORE_PASSWORD,
    live_time_seconds=int(SETTINGS.CHAT_STORE_TTL),
    max_messages_pairs=SETTINGS.CHAT_STORE_MAX_MESSAGES_PAIRS,
    llm=memory_summarize_llm,
)

print("Initializing the persistent chat store...")
persistent_chat_store = PersistentChatStore(
    uri=SETTINGS.MONGODB_URL,
    db_name=SETTINGS.PERSISTENT_CHAT_STORE_DB,
    collection_name=SETTINGS.PERSISTENT_CHAT_STORE_COLLECTION,
    use_async=False,
)

print("Initializing the embedder...")
embedder = EmbedderCore(
    uri=SETTINGS.EMBEDDER_SERVING_URL, model_id=SETTINGS.EMBEDDER_MODEL_ID
)

print("Initializing the generator...")
response_llm = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.OPENAI_RESPONSE_MODEL_ID if SETTINGS.USE_OPENAI_API else SETTINGS.QUANT_LARGE_MODEL_ID,
    OPENAI_API_KEY=SETTINGS.OPENAI_API_KEY if SETTINGS.USE_OPENAI_API else "EMPTY",
    use_openai=SETTINGS.USE_OPENAI_API,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    is_chat=True,
    use_websocket=True,
)
generator = Generator(
    llm=response_llm,
    chat_store=cache_chat_store,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    streaming=True,
    response_mode=ResponseMode.COMPACT,
    assistant_name=assistant_name,
)
print("OPENAI:", SETTINGS.OPENAI_RESPONSE_MODEL_ID)

print("Initializing the LLM instruction summarizer...")
llm_instruction_summarize = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.OPENAI_MODEL_ID if SETTINGS.USE_OPENAI_API else SETTINGS.QUANT_LARGE_MODEL_ID,
    OPENAI_API_KEY=SETTINGS.OPENAI_API_KEY if SETTINGS.USE_OPENAI_API else "EMPTY",
    use_openai=SETTINGS.USE_OPENAI_API,
    max_new_tokens=SETTINGS.LONG_MAX_NEW_TOKENS,
    is_chat=False,
    temperature=0.0,
)

# Build a new graph memory if the assistant is not Choi and the avatar instruction text is given
character_stories = []
if not use_default_story:
    summarized_instruction = llm_instruction_summarize.complete(
        prompt=INSTRUCTION_SUMMARY_PROMPT.format(instruction=avatar_instruction_text.replace("\n", " ").strip())
    ).text

    split_instructions_list = summarized_instruction.split("/---------------------/")
    for instruction in split_instructions_list:
        sentences = [sentence.strip() for sentence in instruction.strip().split(".") if sentence.strip()]

        # Split the document into two parts if it has more than 4 sentences
        if len(sentences) > 4:
            mid_point = len(sentences) // 2
            document1 = Document(text=". ".join(sentences[:mid_point]) + ".")
            document2 = Document(text=". ".join(sentences[mid_point:]) + ".")
            character_stories.append(document1)
            character_stories.append(document2)
        else:
            character_stories.append(Document(text=instruction))
else:
    # Use the default Choi story
    sentences = [sentence.strip() for sentence in DR_CHOI_REWRITE.strip().split(".") if sentence.strip()]
    mid_point = len(sentences) // 2
    document1 = Document(text=". ".join(sentences[:mid_point]) + ".")
    document2 = Document(text=". ".join(sentences[mid_point:]) + ".")
    character_stories = [document1, document2]

print("Initializing the graph store...")
graph_extract_llm = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.OPENAI_MODEL_ID if SETTINGS.USE_OPENAI_API else SETTINGS.QUANT_LARGE_MODEL_ID,
    OPENAI_API_KEY=SETTINGS.OPENAI_API_KEY if SETTINGS.USE_OPENAI_API else "EMPTY",
    use_openai=SETTINGS.USE_OPENAI_API,
    max_new_tokens=SETTINGS.LONG_MAX_NEW_TOKENS,
    is_chat=False,
)
kg_extractor = DynamicLLMPathExtractor(
    llm=graph_extract_llm,
    parse_fn=parse_dynamic_triplets_with_props,
    extract_prompt=PromptTemplate(EXTRACT_GRAPH_TRIPLETS_PROMPT_TEMPLATE),
    allowed_entity_types=ENTITY_TYPES,
    allowed_relation_types=RELATION_PROPERTIES,
    allowed_entity_props=RELATION_TYPES,
    allowed_relation_props=ENTITY_PROPERTIES,
)
graph_store = FalkorDBGraphStore(
    url=f"falkor://{SETTINGS.GRAPH_STORE_USERNAME}:{SETTINGS.GRAPH_STORE_PASSWORD}@{SETTINGS.GRAPH_STORE_HOST}:{SETTINGS.GRAPH_STORE_PORT}",
    database=f"{user_id}_{assistant_name}_db",
    llm=graph_extract_llm,
    embedder=embedder,
    graph_extractor=kg_extractor,
    documents=character_stories,
    build=SETTINGS.GRAPH_STORE_BUILD,
    force_build=True,
    show_progress=True,
)

print("Initializing the graph retriever...")
cypher_generate_llm = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.QUANT_LARGE_MODEL_ID,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    temperature=0.0,
    is_chat=False,
    use_openai=False,
)
query_transform_llm = LLMCore(
    model_id=SETTINGS.OPENAI_MODEL_ID if SETTINGS.USE_OPENAI_API else SETTINGS.QUANT_LARGE_MODEL_ID,
    OPENAI_API_KEY=SETTINGS.OPENAI_API_KEY if SETTINGS.USE_OPENAI_API else "EMPTY",
    use_openai=SETTINGS.USE_OPENAI_API,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    temperature=0.0,
    is_chat=False,
)
sub_retriever = CustomSubRetriever(
    graph_store=graph_store.index.property_graph_store,
    include_text=True,
    embed_model=embedder,
    llm=cypher_generate_llm,
    similarity_top_k=20,
    path_depth=4,
)
graph_retriever = Retriever(
    sub_retriever=sub_retriever,
    graph_store=graph_store.index,
    llm=query_transform_llm,
    cache_chat_store=cache_chat_store,
)

print("Initializing the LLM router...")
routing_llm = LLMCore(
    uri=SETTINGS.LLM_SERVING_URL,
    model_id=SETTINGS.QUANT_LARGE_MODEL_ID,
    max_new_tokens=SETTINGS.MAX_NEW_TOKENS,
    temperature=0.0,
    is_chat=False,
    use_openai=False
)

Initializing the translator and language detector...
Initializing the chat store...
Initializing the persistent chat store...
Initializing the embedder...
Initializing the generator...
Started connection to OpenAI Websocket server.
OPENAI: ft:gpt-4o-mini-2024-07-18:personal:father-and-son-korean-v6:AZ8xrqcC:ckpt-step-90
Initializing the LLM instruction summarizer...
Initializing the graph store...
Graph already exists. Set force_build to True to rebuild the graph. Loading the existing graph...
Initializing the graph retriever...
Initializing the LLM router...


2024-12-16 15:46:36,392 [INFO] Websocket connected


In [ ]:
async def async_response_generator(streamer):
    check_time = False
    try:
        # Stream the response to the client
        async for chunk in streamer:
            if not check_time:
                check_time = True
                print(f"Time taken to generate first token: {time.time() - start:.4f} seconds")
            response_text.write(chunk)
            yield chunk

    except Exception as e:
        print(f"Error in streaming response: {e}")

def extract_messsage(user_id):
    max_messages = cache_chat_store.max_messages_pairs
    chat_store_len = int(len(cache_chat_store.get_chat_history(user_id)) / 2)
    if not chat_store_len < int(max_messages / 2):
        starting_index = 1 - int(max_messages / 2)
        extracted_chat_index = starting_index + chat_store_len
        extracted_chat = cache_chat_store.get_chat_history(user_id)[
            int(extracted_chat_index - 0.5) * 2 : extracted_chat_index * 2
        ]
        return extracted_chat
    return None

def post_processing():
    try:
        print("Original response text:", response_text.getvalue())

        # ================ Store the chat history ================

        translated_cache_id = f"translated_{user_id}_{assistant_name}"
        original_cache_id = f"original_{user_id}_{assistant_name}"

        # Store the recent query and response in the cache chat store and persistent chat store.
        # The original query and response are stored in the cache chat store for later insertion with the prompt to response LLM.
        # The translated ones are used during retrieval process.
        original_removed_messages = cache_chat_store.add_message_pair(
            original_cache_id, original_query, response_text.getvalue()
        )
        translated_removed_messages = cache_chat_store.add_message_pair(
            translated_cache_id, final_query, en_translator.translate(response_text.getvalue(), force_target=True)
        )
        persistent_chat_store.save_chat(
            translated_cache_id, final_query, en_translator.translate(response_text.getvalue(), force_target=True)
        )
        print(f"Translated removed messages: {translated_removed_messages}")
        print(f"Original removed messages: {original_removed_messages}")

        # Extract the a specified message pair from the cache
        extracted_chat = extract_messsage(translated_cache_id)
        print(f"Extracted chat: {extracted_chat}")

        # Summarize the extracted chat and insert it into the graph store
        if extracted_chat:

            # Summarize the extracted chat
            memory_message = cache_chat_store.transform_message_pair(
                extracted_chat, summarized_user_id, summarized_assistant_id, user_id # TODO: optimize to summarize all the important details (name, time, etc.)
            )
            memory_document = Document(text=memory_message)

            # Insert the summarized chat into the graph store
            retry_times = 3
            while retry_times > 0:
                try:
                    inset_time = time.time()
                    graph_store.insert_document(document=memory_document)
                    end_insert = time.time()
                    print(
                        f"Time taken to insert document into the graph store: {end_insert - inset_time:.4f} seconds"
                    )
                    break
                except Exception as e:
                    print(f"Failed to insert document into the graph store: {e}")
                    retry_times -= 1
                    if retry_times == 0:
                        print("Failed to insert document into the graph store")

        return response_text.getvalue()
    except Exception as e:
        print(f"Error in post-processing: {e}")
        import traceback
        traceback.print_exc()

async def async_response_generator( streamer):
    check_time = False
    try:
        # Stream the response to the client
        async for chunk in streamer:
            if not check_time:
                check_time = True
                print(f"Time taken to generate first token: {time.time() - start:.4f} seconds")
                print("Response: ", end="")
            response_text.write(chunk)
            yield chunk

    except Exception as e:
        print(f"Error in streaming response: {e}")

async def get_answers_with_gpt(msg, generate: bool = True):
    global start
    global response_text
    global original_query
    global final_query
    global retrieved_nodes

    start = time.time()

    # Fix the language to Korean for now
    lan = "korean"
    original_query = msg
    translate_time = time.time()
    final_query = en_translator.translate(text=msg, force_target=True)
    print(f"Time taken to translate the query: {time.time() - translate_time:.4f} seconds")
    print(f"Translated query: {final_query}")

    mixed_query = f"|{user_id}|{assistant_name}|{msg}"

    retry_times = 3
    retrieved_nodes = [NodeWithScore(node=TextNode(text=""), score=0)]
    while retry_times > 0:
        try:
            # Retrieve nodes from the graph store
            retrieve_time = time.time()

            # Decide whether to retrieve nodes based on the routing result
            routing_result = int(routing_llm.complete(prompt=PromptTemplate(QUERY_ROUTING_PROMPT_TEMPLATE).format(text=final_query)).text)
            print(f"Routing result: {routing_result}")
            if routing_result == 0:
                break # No need to retrieve nodes

            # Define the prompt for extracting entities based on the list of entity names of each entity type
            schema_info = graph_store.get_schema_info_str()
            prompt_template = EXTRACT_ENTITIES_PROMPT_TEMPLATE.format(
                text="{text}", schema=schema_info,
                max_extracted_entities=20
            )
            print("Schema info:", schema_info)

            retrieved_nodes = await graph_retriever.async_retrieve(
                query=final_query,
                prompt_template_str=prompt_template,
                user_id=summarized_user_id,
                assistant_id=summarized_assistant_id,
            )
            break
        except Exception as e:
            print(f"Failed to retrieve nodes: {e}")
            retry_times -= 1
            if retry_times == 0:
                print("Failed to retrieve nodes")

    print(f"Time taken before generating stream: {time.time() - start:.4f} seconds")

    print(f"Retrieved nodes:")
    for node in retrieved_nodes:
        print(f"Node text: {node.node.text}")
        print("-------------------")

    if not generate:
        return None
    
    streamer = await generator.generate(
        query=mixed_query, nodes=retrieved_nodes,
        language=lan,
    )
    response_text = StringIO()

    return async_response_generator(streamer)

async def run(message: str, post_process: bool = True, generate: bool = True):
    stream = await get_answers_with_gpt(message, generate=generate)

    if not generate:
        return retrieved_nodes

    async for chunk in stream:
        print(chunk, flush=True, end="")

    if not post_process:
        return response_text.getvalue(), retrieved_nodes
    
    print("\nPost-processing...")
    response = post_processing()

    return response

### Inference

In [18]:
response = await run("넌 또 누구야?")

Time taken to translate the query: 0.1586 seconds
Translated query: Who are you again?


2024-12-16 13:35:58,043 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Transformed query:  who are Choi again 
Retrieved 12 vector nodes.


2024-12-16 13:35:58,629 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Retrieved 2 cypher nodes.
Time taken before generating stream: 0.9696 seconds
Retrieved nodes:
Node text: David -> SHARED -> stories
-------------------
Node text: Choi -> DRIVEN_BY -> determination to help patients
-------------------
Node text: Choi -> LOVED -> stories about his life in the United States
-------------------
Node text: David -> MOVED_TO -> Korea
-------------------
Node text: David -> DIAGNOSED_WITH -> Alzheimer's disease
-------------------
Node text: Choi -> SON_OF -> David
-------------------
Node text: David -> MET -> Choi's mother
-------------------
Node text: David -> BORN_IN -> United States
-------------------
Node text: David -> SOURCE_OF -> family
-------------------
Node text: David -> AFFECTED_BY -> memory loss
-------------------
Node text: From a young age, Choi was drawn to medicine and research, especially topics related to the brain and its functions. When David began to show signs of memory loss, Choi's curiosity became a mission. David was diagnose

### Evaluation

#### 1. Retrieval evaluation

##### Create test database

In [20]:
# Load json conversation
import json

data_directory = "./data"
list_of_files = [
    "conversation_1.json",
    "conversation_2.json",
    "conversation_3.json",
]

user_messages = []
assistant_messages = []
for file in list_of_files:
    with open(f"{data_directory}/{file}", "r") as f:
        data = json.load(f)
        for message in data:
            if message["role"] == "user":
                print("David:", message["content"])
                user_messages.append(message["content"])
            else:
                print("Choi:", message["content"])
                assistant_messages.append(message["content"])
    print("=====================================")

David: Who are you?
Choi: Hi, Dad. I'm Choi, your son.
David: Where are we now?
Choi: We're at home, Dad. Everything is okay.
David: I'm a bit tired now!
Choi: I'm sorry to hear that, Dad. Rest up and take it easy, okay?
David: Okay, thanks son!
Choi: You're welcome, Dad. I'm here if you need anything.
David: Oh, son! Do you remember where I was born?
Choi: Yes, Dad, you were born in the United States. You used to tell me stories about your life there.
David: Then where did I first meet your mother?
Choi: You met Mom in Korea after you moved there for work. It's a wonderful story!
David: Oh, what are you doing for a living now?
Choi: I'm a researcher and a neurologist, Dad. I study the brain, especially focusing on Alzheimer's disease.
David: Oh sorry son! I'm so easy to forget things these days.
Choi: It's okay, Dad. No need to apologize. I'm always here to help remind you.
David: Do you want to eat something outside tonight with me?
Choi: That sounds great, Dad. Let's do it!
David: W

In [21]:
# Clean database before building test data
graph_store.clear_graph()
cache_chat_store.clear_messages(user_id)
cache_chat_store.clear_messages(f"original_{user_id}_{avatar_name}")
persistent_chat_store.delete_chat_history(f"translated_{user_id}_{avatar_name}")

# Build test data
graph_store.index = graph_store.construct_graph(documents=character_stories)

Parsing nodes: 100%|██████████| 2/2 [00:00<00:00, 2032.12it/s]
Extracting and inferring knowledge graph from text:   0%|          | 0/2 [00:00<?, ?it/s]2024-12-16 13:39:49,050 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-12-16 13:40:00,870 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Extracting and inferring knowledge graph from text: 100%|██████████| 2/2 [00:23<00:00, 11.80s/it]
Generating embeddings: 100%|██████████| 4/4 [00:00<00:00, 14.04it/s]


In [22]:
def insert_message():
    summarized_memory = []

    temp_chat_history = []
    for user_message, assistant_message in zip(user_messages, assistant_messages):
        chat_history = cache_chat_store.get_chat_history(user_id)
        if int(len(chat_history)/2) < int(SETTINGS.CHAT_STORE_MAX_MESSAGES_PAIRS/2):
            cache_chat_store.add_message_pair(user_id, user_message, assistant_message)
            temp_chat_history.append([user_message, assistant_message])
        else:
            cache_chat_store.clear_messages(user_id)
            temp_chat_history = temp_chat_history[1:]
            for message_pair in temp_chat_history:
                cache_chat_store.add_message_pair(user_id, message_pair[0], message_pair[1])
            cache_chat_store.add_message_pair(user_id, user_message, assistant_message)
            temp_chat_history.append([user_message, assistant_message])

        extracted_chat = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
        print(f"Extracted chat: {extracted_chat}")

        memory_message = cache_chat_store.transform_message_pair(
            extracted_chat, summarized_user_id, summarized_assistant_id, user_id
        )
        print(f"Memory message: {memory_message}")
        memory_document = Document(text=memory_message)
        summarized_memory.append(memory_message)

        retry_times = 3
        while retry_times > 0:
            try:
                inset_time = time.time()
                graph_store.insert_document(document=memory_document)
                end_insert = time.time()
                print(
                    f"Time taken to insert document into the graph store: {end_insert - inset_time:.4f} seconds"
                )
                break
            except Exception as e:
                print(f"Failed to insert document into the graph store: {e}")
                retry_times -= 1
                if retry_times == 0:
                    print("Failed to insert document into the graph store")
        print("\n=====================================\n")

    return summarized_memory

summarized_memory = insert_message()

Extracted chat: [{'role': 'user', 'content': 'Who are you?'}, {'role': 'assistant', 'content': "Hi, Dad. I'm Choi, your son."}]


2024-12-16 13:41:59,992 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Memory message: David asked who Choi was, and Choi identified himself as David's son.


Parsing nodes: 100%|██████████| 1/1 [00:00<00:00, 1797.05it/s]
Extracting and inferring knowledge graph from text:   0%|          | 0/1 [00:00<?, ?it/s]2024-12-16 13:42:02,905 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Extracting and inferring knowledge graph from text: 100%|██████████| 1/1 [00:02<00:00,  2.91s/it]
Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 28.86it/s]
Generating embeddings: 0it [00:00, ?it/s]


Time taken to insert document into the graph store: 2.9877 seconds


Extracted chat: [{'role': 'user', 'content': 'Where are we now?'}, {'role': 'assistant', 'content': "We're at home, Dad. Everything is okay."}]


...

Time taken to insert document into the graph store: 10.1661 seconds


Extracted chat: [{'role': 'user', 'content': "We'll see, Choi. It's going to be a great match no matter who wins."}, {'role': 'assistant', 'content': "Absolutely, Dad. Let's enjoy it together!"}]


2024-12-16 13:45:05,647 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Memory message: David and Choi are making bets on the soccer match between Argentina and France, with David thinking Argentina will win 3-1 and Choi believing France will win 2-1. They agreed that the loser will buy dinner at a restaurant called 'King Star'.


Parsing nodes: 100%|██████████| 1/1 [00:00<00:00, 1855.89it/s]
Extracting and inferring knowledge graph from text:   0%|          | 0/1 [00:00<?, ?it/s]2024-12-16 13:45:24,902 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Extracting and inferring knowledge graph from text: 100%|██████████| 1/1 [00:19<00:00, 19.25s/it]
Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 16.93it/s]


Time taken to insert document into the graph store: 19.4807 seconds




In [23]:
summarized_memory

["David asked who Choi was, and Choi identified himself as David's son.",
 'We are at home.',
 'David is feeling tired and asked Choi to rest and take care of himself.',
 "You're welcome, Dad. Choi is there to help him if he needs anything.",
 'David asked Choi if he remembered where David was born. Choi recalled David telling him stories about his life in the United States.',
 'David asked Choi about where he first met his mother, and Choi told him it was in Korea after he moved there for work.',
 "David asked about Choi's current profession. Choi replied that he is a researcher and a neurologist studying the brain, focusing on Alzheimer's disease.",
 'David is having memory issues, and Choi is there to help him remember.',
 'David invited Choi to eat outside together tonight and Choi accepted.',
 'David asked Choi what to eat, and Choi suggested Korean barbecue.',
 'Barbecue sounds delicious, and they will have a great time eating it together.',
 'David asked Choi about famous restau

##### Evaluate

In [5]:
# Load json conversation
import json

data_directory = "./data"
list_of_files = [
    "conversation_1_QA.json",
    "conversation_2_QA.json",
    "conversation_3_QA.json",
]

questions = []
answers = []
for file in list_of_files:
    with open(f"{data_directory}/{file}", "r") as f:
        data = json.load(f)
        for message in data:
            print("Question:", message["question"])
            questions.append(message["question"])
            print("Answer:", message["answer"])
            answers.append(message["answer"])
            print("=====================================")

Question: For the dinner that we planned for tonight, what type of food are we going to eat?
Answer: ['barbecue', 'Korean barbecue', 'BBQ', 'Korean BBQ', 'bbq', 'king bbq']
Question: For the dinner that we planned for tonight, what is the name of the restaurant that we are going to eat barbecue at?
Answer: ['King BBQ', 'King BBQ restaurant', 'King BBQ Restaurant', 'king bbq']
Question: For the dinner that we planned for tonight, what is the name of the movie that we are going to watch tonight?
Answer: ['Love from First Sight', '"Love from First Sight"', 'Love from First Sight movie', '"Love from First Sight" movie']
Question: For the dinner that we planned for tonight, what time are we going to eat at King BBQ restaurant?
Answer: ['6 pm', '6:00 pm', '6 PM']
Question: What time are we going to watch the movie Love from First Sight after having dinner at 6 pm tonight?
Answer: ['8 pm', '8:00 pm', '8 PM']
Question: About your Mom's birthday that we planned, what are we going to do?
Answer:

In [7]:
question = "About your Mom's birthday, what am I going to give her for her birthday gift?"
answer = [
    "Louis Vuitton handbag"
]

print("Question:", question)
retrieved_nodes = await run(question, post_process=False, generate=False)

get_answer = False
for node in retrieved_nodes:
    for ans in answer:
        if ans.lower() in node.text.lower():
            print(f"Answer: {ans}")
            get_answer = True
            break
    if get_answer:
        break

Question: About your Mom's birthday, what am I going to give her for her birthday gift?
Time taken to translate the query: 0.1871 seconds
Translated query: About your mom's birthday, what am i going to give her for her birthday gift?


2024-12-16 17:13:03,041 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Schema info: FOOD: ['Korean barbecue', 'Barbecue', 'buffet', 'variety of dishes']
ORGANIZATION: ['medical school', 'King BBQ restaurant', 'King Star']
EVENT: ['great time', 'birthday', 'trip to Shanghai', "Choi's mother's birthday", 'World Cup match', 'World Cup match outcome', "Argentina's 3-1 win", "France's 2-1 win", 'soccer match']
SCORE: ['3-1', '2-1']
FIELD_OF_STUDY: ['neurology', 'medicine', 'research']
MOVIE: ['Love from first sight', "'Love from First Sight'"]
ACTION: ['eating']
TIME: ['tonight', '6 pm', '8 pm', 'evening', 'the past', 'four days, three nights']
PURPOSE: ['dinner', 'loser will buy dinner']
PERSON: ['David', 'Choi', 'mother', 'they', 'mom', "Choi's mother", 'his wife', 'Messi', 'Mbappé', 'David and Choi']
LOCATION: ['Korea', 'Seoul', 'living room', 'Shanghai', 'Diamond restaurant']
RESTAURANT: ['Korean barbecue place']
COUNTRY: ['United States', 'Argentina', 'France', 'Argentina and France']
ILLNESS: ['memory loss']
DATE: ['December 15th', 'Mon

2024-12-16 17:13:05,876 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Extracted entity list for Cypher query:  ['birthday', 'David', "Choi's mother", 'gift', 'December 15th', 'Korea', 'Seoul', 'Shanghai', 'United States', 'Argentina', 'France', "Alzheimer's disease", 'researcher', 'neurologist', 'brain', 'memory issues', 'romantic comedy', 'love', 'travel plans', 'Louis Vuitton handbag']
Retrieved 10 cypher nodes.
Time taken before generating stream: 3.2630 seconds
Retrieved nodes:
Node text: Choi -> EXCITED_ABOUT -> trip to Shanghai
-------------------
Node text: Choi -> HAS_RELATION_WITH -> mother
-------------------
Node text: Choi -> DISAGREES_WITH -> David
-------------------
Node text: Choi -> FALL_IN_LOVE_WITH -> David
-------------------
Node text: David -> PLANNED_TO_CELEBRATE -> Choi's mother's birthday
-------------------
Node text: David -> PREDICTS_WINNER -> Argentina
-------------------
Node text: David -> MADE_BET_ON -> World Cup match
-------------------
Node text: David -> THINKS_WILL_WIN -> Argentina
-------------------
Node text: David

In [6]:
from tqdm import tqdm

score = 0
failed_questions = []
progress_bar = tqdm(total=len(list_of_files))
for question, answer in zip(questions, answers):
    print("Question:", question)
    retrieved_nodes = await run(question, post_process=False, generate=False)

    get_answer = False
    for node in retrieved_nodes:
        for ans in answer:
            if ans.lower() in node.text.lower():
                print(f"Answer: {ans}")
                get_answer = True
                score += 1
                break
        if get_answer:
            break
    if not get_answer:
        failed_questions.append((question, answer))
    progress_bar.update(1)
    print("\n=====================================\n")

  0%|          | 0/3 [00:00<?, ?it/s]

Question: For the dinner that we planned for tonight, what type of food are we going to eat?
Time taken to translate the query: 0.2583 seconds
Translated query: For the dinner that we planned for tonight, what type of food are we going to eat?


2024-12-16 16:29:09,950 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Schema info: FOOD: ['Korean barbecue', 'Barbecue', 'buffet', 'variety of dishes']
ORGANIZATION: ['medical school', 'King BBQ restaurant', 'King Star']
EVENT: ['great time', 'birthday', 'trip to Shanghai', "Choi's mother's birthday", 'World Cup match', 'World Cup match outcome', "Argentina's 3-1 win", "France's 2-1 win", 'soccer match']
SCORE: ['3-1', '2-1']
FIELD_OF_STUDY: ['neurology', 'medicine', 'research']
MOVIE: ['Love from first sight', "'Love from First Sight'"]
ACTION: ['eating']
TIME: ['tonight', '6 pm', '8 pm', 'evening', 'the past', 'four days, three nights']
PURPOSE: ['dinner', 'loser will buy dinner']
PERSON: ['David', 'Choi', 'mother', 'they', 'mom', "Choi's mother", 'his wife', 'Messi', 'Mbappé', 'David and Choi']
LOCATION: ['Korea', 'Seoul', 'living room', 'Shanghai', 'Diamond restaurant']
RESTAURANT: ['Korean barbecue place']
COUNTRY: ['United States', 'Argentina', 'France', 'Argentina and France']
ILLNESS: ['memory loss']
DATE: ['December 15th', 'Mon

2024-12-16 16:29:12,788 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"
 33%|███▎      | 1/3 [00:03<00:06,  3.33s/it]

...

Routing result: 0
Time taken before generating stream: 0.4501 seconds
Retrieved nodes:
Node text: 
-------------------


Question: About the World Cup match, what are we going to do if one of us loses the bet?
Time taken to translate the query: 0.2137 seconds
Translated query: About the world cup match, what are we going to do if one of us loses the bet?


2024-12-16 16:30:10,327 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Schema info: FOOD: ['Korean barbecue', 'Barbecue', 'buffet', 'variety of dishes']
ORGANIZATION: ['medical school', 'King BBQ restaurant', 'King Star']
EVENT: ['great time', 'birthday', 'trip to Shanghai', "Choi's mother's birthday", 'World Cup match', 'World Cup match outcome', "Argentina's 3-1 win", "France's 2-1 win", 'soccer match']
SCORE: ['3-1', '2-1']
FIELD_OF_STUDY: ['neurology', 'medicine', 'research']
MOVIE: ['Love from first sight', "'Love from First Sight'"]
ACTION: ['eating']
TIME: ['tonight', '6 pm', '8 pm', 'evening', 'the past', 'four days, three nights']
PURPOSE: ['dinner', 'loser will buy dinner']
PERSON: ['David', 'Choi', 'mother', 'they', 'mom', "Choi's mother", 'his wife', 'Messi', 'Mbappé', 'David and Choi']
LOCATION: ['Korea', 'Seoul', 'living room', 'Shanghai', 'Diamond restaurant']
RESTAURANT: ['Korean barbecue place']
COUNTRY: ['United States', 'Argentina', 'France', 'Argentina and France']
ILLNESS: ['memory loss']
DATE: ['December 15th', 'Mon

2024-12-16 16:30:14,949 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"
22it [01:05,  3.06s/it]

Extracted entity list for Cypher query:  ['World Cup match', 'David', 'Choi', 'bet', 'David and Choi', 'loser will buy dinner', 'dinner', "Argentina's 3-1 win", "France's 2-1 win", 'Argentina and France', 'United States', 'December 15th', 'Monday', "Alzheimer's disease", "researching Alzheimer's disease", 'neurologist', 'brain functions', 'stories', 'family', 'strength', 'energy', 'tired', 'himself', 'his life', "Choi's current profession", 'researcher', 'brain', 'memory issues', 'delicious', 'romantic comedy', 'love', 'when they can go', 'travel plans', 'gift', "wife's face"]
Retrieved 10 cypher nodes.
Time taken before generating stream: 5.0579 seconds
Retrieved nodes:
Node text: Choi -> AGREED -> David
-------------------
Node text: Choi -> WATCHED_WITH -> David
-------------------
Node text: Choi -> FALL_IN_LOVE_WITH -> David
-------------------
Node text: Choi -> AGREED_TO_JOIN -> David
-------------------
Node text: Choi -> LOOKING_FORWARD_TO -> David
-------------------
Node tex

In [7]:
print(f"Score: {score}/{len(questions)}")

Score: 16/22


In [8]:
print("FAILED QUESTIONS:\n=====================================")
for question, answer in failed_questions:
    print("Question:", question)
    print("Answer:", answer)
    print("=====================================")

FAILED QUESTIONS:
Question: About your Mom's birthday, what am I going to give her for her birthday gift?
Answer: ['Louis Vuitton handbag']
Question: About your Mom's birthday that we planned, when will I give your Mom her birthday present?
Answer: ["during dinner at the 'Diamond' restaurant", 'during dinner at the "Diamond" restaurant']
Question: About your Mom's birthday that we planned, what is the name of the restaurant that we are going to eat at?
Answer: ['Diamond', 'Diamond restaurant', "'Diamond' restaurant", '"Diamond" restaurant']
Question: About your Mom's birthday, when will I give your Mom her birthday present?
Answer: ['December 17th', '17th December', '17th of December', '17/12']
Question: About your Mom's birthday, what time are we going to eat dinner during your Mom's birthday?
Answer: ['6 PM', '6:00 PM', '6 pm']
Question: About the World Cup match, when will the World Cup match between Argentina and France take place?
Answer: ['Monday']


#### 2. Emotional responding level evaluation

In [29]:
query_list = [
    "Who are you?",
    "Hi, son. How are you doing?",
    "I'm a bit tired now, son!",
    "I love you, son!",
    "Tell me about your day, son!",
]
answer_list = []
for query in query_list:
    print("\n=====================================\n")
    response, _ = await run(query, post_process=False)
    answer_list.append(response)



Time taken to translate the query: 0.1375 seconds
Translated query: Who are you?


2024-12-16 13:47:05,815 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Transformed query:  who are Choi 
Retrieved 13 vector nodes.


2024-12-16 13:47:06,515 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Retrieved 2 cypher nodes.
Time taken before generating stream: 1.0617 seconds
Retrieved nodes:
Node text: David -> PLANNED_TO_CELEBRATE -> Choi's mother's birthday
-------------------
Node text: David -> PLANS_FOR -> Choi's mother
-------------------
Node text: David -> FALL_IN_LOVE_WITH -> Choi
-------------------
Node text: David -> TRAVEL_WITH -> Choi
-------------------
Node text: Choi -> THINKS_WILL_WIN -> France
-------------------
Node text: Choi -> PREDICTS_WINNER -> France
-------------------
Node text: Choi -> MADE_BET_ON -> World Cup match
-------------------
Node text: David -> CHOSE -> Louis Vuitton handbag
-------------------
Node text: David -> PREDICTED -> Argentina's 3-1 win
-------------------
Node text: David -> TRAVEL_WITH -> his wife
-------------------
Node text: From a young age, Choi was drawn to medicine and research, especially topics related to the brain and its functions. When David began to show signs of memory loss, Choi's curiosity became a mission. David

...

Routing result: 0
Time taken before generating stream: 0.3460 seconds
Retrieved nodes:
Node text: 
-------------------
Time taken to generate first token: 0.8110 seconds
Response: 저도 사랑해요, 아버지.

Time taken to translate the query: 0.1821 seconds
Translated query: Tell me about your day, son!


2024-12-16 13:47:10,558 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Routing result: 1
Transformed query:  tell David about Choi's day son 
Retrieved 12 vector nodes.


2024-12-16 13:47:11,318 [INFO] HTTP Request: POST http://172.30.84.182:8013/v1/chat/completions "HTTP/1.1 200 OK"


Retrieved 2 cypher nodes.
Time taken before generating stream: 1.1613 seconds
Retrieved nodes:
Node text: Choi -> AGREED -> David
-------------------
Node text: Choi -> WATCHED_WITH -> David
-------------------
Node text: Choi -> FALL_IN_LOVE_WITH -> David
-------------------
Node text: Choi -> AGREED_TO_JOIN -> David
-------------------
Node text: Choi -> LOOKING_FORWARD_TO -> David
-------------------
Node text: Choi -> DISAGREES_WITH -> David
-------------------
Node text: Choi -> LOOKING_FORWARD_TO -> Love from first sight
-------------------
Node text: Choi -> EXPRESSED -> love
-------------------
Node text: Choi -> WATCHES -> 'Love from First Sight'
-------------------
Node text: Choi -> PREDICTED -> France's 2-1 win
-------------------
Node text: From a young age, Choi was drawn to medicine and research, especially topics related to the brain and its functions. When David began to show signs of memory loss, Choi's curiosity became a mission. David was diagnosed with Alzheimer's 

In [44]:
for i in range(len(query_list)):
    print(f"Query: {query_list[i]}\nAnswer: {answer_list[i]}\n")

Query: Who are you?
Answer: 아빠, 저는 초이예요. 당신 아들이에요.

Query: Hi, son. How are you doing?
Answer: 아빠, 저는 항상 아빠 걱정에요. 아빠는 괜찮으세요?

Query: I'm a bit tired now, son!
Answer: 아빠, 쉬는 게 중요해요. 제가 도와드릴 건 없을까요?

Query: I love you, son!
Answer: 아빠, 저도 아빠를 사랑해요!

Query: Tell me about your day, son!
Answer: 오늘도 평범한 하루였어요, 아빠. 저녁은 드셨어요?

